# 🔧 AI-Integrated CAD Design Validation System

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/princenishad5471/Projects_Resume_Tnp/blob/main/AI_CAD_Design_Validation.ipynb)

## Overview
This notebook implements an **AI-Integrated CAD Assistance** system that:

- ✅ Automatically validates CAD designs against established engineering standards  
- 🤖 Uses an **Isolation Forest** AI model to detect anomalous design patterns  
- 📊 Generates detailed validation reports with severity-based issue classification  
- 📈 Visualises design deviations, compliance statistics, and anomaly scores  

---
### Problem Addressed
Manual design verification is time-consuming, error-prone, and expertise-dependent.  
This system enables **early detection** of design issues during the design phase,  
reducing rework, project delays, and improving overall design quality.


## 1. Importing Dependencies

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler
from datetime import datetime
import warnings
warnings.filterwarnings("ignore")

# Import the CAD validation module
from cad_design_validator import (
    DESIGN_STANDARDS,
    DesignStandardsChecker,
    AIAnomalyDetector,
    CADValidationEngine,
    ValidationReportGenerator,
)

# Plot styling consistent with existing project notebooks
sns.set_style("whitegrid")
plt.rcParams.update({"figure.dpi": 100, "figure.figsize": (10, 5)})
print("✅ All dependencies loaded successfully.")


## 2. Simulated CAD Design Dataset

We simulate a **historical database of 500 compliant designs** (used to train the AI anomaly detector)  
and a **batch of 50 new designs under review** (some containing intentional non-compliances).


In [ ]:
np.random.seed(42)
N_HIST = 500   # historical compliant designs
N_NEW  = 50    # new designs to validate

# ── Historical compliant designs (training data) ──────────────────────────
historical_designs = pd.DataFrame({
    "length_mm":           np.random.normal(100.0, 4.0,  N_HIST).clip(85,  115),
    "width_mm":            np.random.normal(50.0,  2.0,  N_HIST).clip(44,   56),
    "height_mm":           np.random.normal(30.0,  1.5,  N_HIST).clip(25,   35),
    "wall_thickness_mm":   np.random.normal(5.0,   0.3,  N_HIST).clip(4.0,  6.0),
    "hole_diameter_mm":    np.random.normal(10.0,  0.5,  N_HIST).clip(8.5,  11.5),
    "fillet_radius_mm":    np.random.normal(2.0,   0.2,  N_HIST).clip(1.5,  2.5),
    "draft_angle_deg":     np.random.normal(2.0,   0.3,  N_HIST).clip(1.0,  3.0),
    "surface_roughness_um":np.random.normal(1.6,   0.2,  N_HIST).clip(0.8,  2.5),
})

# ── New designs under review ──────────────────────────────────────────────
new_designs = pd.DataFrame({
    "design_id":           range(1, N_NEW + 1),
    "component_name":      [f"Component_{i:03d}" for i in range(1, N_NEW + 1)],
    "length_mm":           np.random.normal(100.0, 5.0, N_NEW),
    "width_mm":            np.random.normal(50.0,  3.0, N_NEW),
    "height_mm":           np.random.normal(30.0,  2.0, N_NEW),
    "wall_thickness_mm":   np.random.normal(5.0,   0.5, N_NEW),
    "hole_diameter_mm":    np.random.normal(10.0,  1.0, N_NEW),
    "fillet_radius_mm":    np.random.normal(2.0,   0.4, N_NEW),
    "draft_angle_deg":     np.random.normal(2.0,   0.5, N_NEW),
    "surface_roughness_um":np.random.normal(1.6,   0.4, N_NEW),
})

# ── Inject deliberate non-compliances ────────────────────────────────────
# Design 5 : wall too thin
new_designs.loc[4, "wall_thickness_mm"] = 0.4
# Design 12: extreme length anomaly
new_designs.loc[11, "length_mm"]        = 6000.0
# Design 20: very small hole
new_designs.loc[19, "hole_diameter_mm"] = 0.3
# Design 35: surface roughness out of range
new_designs.loc[34, "surface_roughness_um"] = 30.0
# Design 42: anomalous width
new_designs.loc[41, "width_mm"]         = 2500.0

print(f"Historical designs : {len(historical_designs)} rows")
print(f"New designs        : {len(new_designs)} rows")
print("\nSample of new designs:")
new_designs.head()


## 3. Training the AI Anomaly Detector

An **Isolation Forest** model is trained on the historical compliant design database.  
It learns the normal distribution of design parameters so it can flag statistical outliers.


In [ ]:
detector = AIAnomalyDetector(contamination=0.05, random_state=42)
detector.train(historical_designs)

print("✅ AI Anomaly Detector trained on", len(historical_designs), "historical designs.")
print("Features used:", detector.features)


## 4. AI Anomaly Detection on New Designs

In [ ]:
numeric_cols = detector.features
ai_results = detector.detect(new_designs[["design_id"] + numeric_cols])

anomalies = ai_results[ai_results["anomaly_label"] == "ANOMALY"]
print(f"Total designs scanned : {len(ai_results)}")
print(f"Anomalies detected    : {len(anomalies)}")
print("\nAnomalous designs:")
anomalies[["design_id", "anomaly_score"] + numeric_cols].round(3)


### 4.1 Anomaly Score Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ── Left: scatter of anomaly scores ──────────────────────────────────────
colors = ai_results["anomaly_label"].map({"NORMAL": "#2196F3", "ANOMALY": "#F44336"})
axes[0].scatter(ai_results["design_id"], ai_results["anomaly_score"],
                c=colors, edgecolors="white", linewidths=0.5, s=80, zorder=3)
axes[0].axhline(0, color="gray", linestyle="--", linewidth=1, label="Decision boundary")
axes[0].set_xlabel("Design ID")
axes[0].set_ylabel("Anomaly Score (higher = more normal)")
axes[0].set_title("AI Anomaly Scores per Design")
normal_patch  = mpatches.Patch(color="#2196F3", label="Normal")
anomaly_patch = mpatches.Patch(color="#F44336", label="Anomaly")
axes[0].legend(handles=[normal_patch, anomaly_patch])
axes[0].grid(True, alpha=0.3)

# ── Right: histogram of scores ────────────────────────────────────────────
axes[1].hist(ai_results[ai_results["anomaly_label"]=="NORMAL"]["anomaly_score"],
             bins=15, color="#2196F3", alpha=0.7, label="Normal")
axes[1].hist(ai_results[ai_results["anomaly_label"]=="ANOMALY"]["anomaly_score"],
             bins=5,  color="#F44336", alpha=0.8, label="Anomaly")
axes[1].set_xlabel("Anomaly Score")
axes[1].set_ylabel("Count")
axes[1].set_title("Score Distribution")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("anomaly_scores.png", bbox_inches="tight")
plt.show()
print("Figure saved to anomaly_scores.png")


## 5. Automated Rule-Based Design Validation

Each new design is checked against the **engineering design standards**:  
dimensional bounds, material requirements, and assembly constraints.


In [ ]:
engine = CADValidationEngine(ai_detector=detector)

# ── Validate each component ───────────────────────────────────────────────
PARAM_COLS = ["length_mm", "width_mm", "height_mm", "wall_thickness_mm",
              "hole_diameter_mm", "fillet_radius_mm", "draft_angle_deg",
              "surface_roughness_um"]

for _, row in new_designs.iterrows():
    dims = {col: row[col] for col in PARAM_COLS}
    # Simulate material and assembly data for first few components
    material = {
        "name": np.random.choice(["Steel", "Aluminum", "Titanium", "ABS", "Plastic"]),
        "yield_strength_mpa": float(np.random.normal(250, 60)),
        "density_g_cm3":      float(np.random.uniform(2.5, 9.0)),
    }
    assembly = {
        "interference_mm": float(np.random.uniform(0, 0.08)),
        "clearance_mm":    float(np.random.uniform(0.05, 0.3)),
        "component_count": int(np.random.randint(10, 600)),
    }
    engine.validate_component(row["component_name"], dims, material, assembly)

summary = engine.summary()
print("Validation complete.")
print(f"  Rule violations  : {summary['total_issues']}")
print(f"  AI anomalies     : {summary['ai_anomalies']}")
print(f"  Overall status   : {'PASSED ✅' if summary['passed'] else 'FAILED ❌'}")


### 5.1 Validation Issues Summary Table

In [ ]:
issues_df = engine.issues_dataframe()
print(f"Total issues found: {len(issues_df)}")
if not issues_df.empty:
    issues_df[["severity","component","parameter","measured_value","expected_value","description"]].head(20)


### 5.2 Issue Severity Breakdown

In [ ]:
if not issues_df.empty:
    sev_order  = ["CRITICAL", "HIGH", "MEDIUM", "LOW", "INFO"]
    sev_colors = {"CRITICAL":"#D32F2F","HIGH":"#F57C00","MEDIUM":"#1976D2",
                  "LOW":"#0097A7","INFO":"#388E3C"}

    sev_counts = (issues_df["severity"]
                  .value_counts()
                  .reindex(sev_order, fill_value=0))

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # ── Bar chart ────────────────────────────────────────────────────────
    bars = axes[0].bar(sev_counts.index,
                       sev_counts.values,
                       color=[sev_colors[s] for s in sev_counts.index],
                       edgecolor="white", linewidth=0.8)
    axes[0].bar_label(bars, padding=3, fontsize=11)
    axes[0].set_xlabel("Severity Level")
    axes[0].set_ylabel("Number of Issues")
    axes[0].set_title("Validation Issues by Severity")
    axes[0].set_ylim(0, sev_counts.max() * 1.2 + 1)

    # ── Pie chart ────────────────────────────────────────────────────────
    non_zero = sev_counts[sev_counts > 0]
    axes[1].pie(non_zero, labels=non_zero.index,
                colors=[sev_colors[s] for s in non_zero.index],
                autopct="%1.1f%%", startangle=140,
                wedgeprops=dict(edgecolor="white", linewidth=1.5))
    axes[1].set_title("Severity Distribution (%)")

    plt.tight_layout()
    plt.savefig("severity_breakdown.png", bbox_inches="tight")
    plt.show()
    print("Figure saved to severity_breakdown.png")
else:
    print("No issues to display.")


### 5.3 Most Frequent Violation Types

In [ ]:
if not issues_df.empty:
    fig, ax = plt.subplots(figsize=(12, 5))

    rule_counts = (issues_df["rule_id"]
                   .value_counts()
                   .head(15)
                   .sort_values())

    colors = []
    for rule in rule_counts.index:
        sev = issues_df.loc[issues_df["rule_id"]==rule, "severity"].iloc[0]
        colors.append(sev_colors.get(sev, "#607D8B"))

    bars = ax.barh(rule_counts.index, rule_counts.values,
                   color=colors, edgecolor="white", linewidth=0.5)
    ax.bar_label(bars, padding=3)
    ax.set_xlabel("Count")
    ax.set_title("Top Violated Design Rules")

    legend_handles = [mpatches.Patch(color=v, label=k) for k, v in sev_colors.items()]
    ax.legend(handles=legend_handles, loc="lower right", fontsize=9)

    plt.tight_layout()
    plt.savefig("violation_types.png", bbox_inches="tight")
    plt.show()
    print("Figure saved to violation_types.png")


### 5.4 Design Parameter Distribution vs Standards

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes = axes.flatten()

params_to_plot = [
    ("length_mm",           "Length (mm)",          85, 115),
    ("width_mm",            "Width (mm)",            44,  56),
    ("height_mm",           "Height (mm)",           25,  35),
    ("wall_thickness_mm",   "Wall Thickness (mm)",    4,   6),
    ("hole_diameter_mm",    "Hole Diameter (mm)",    8.5, 11.5),
    ("fillet_radius_mm",    "Fillet Radius (mm)",    1.5,  2.5),
    ("draft_angle_deg",     "Draft Angle (°)",        1,   3),
    ("surface_roughness_um","Surface Roughness (µm)", 0.8, 2.5),
]

for ax, (col, label, lo, hi) in zip(axes, params_to_plot):
    data = new_designs[col]
    color = np.where((data < lo) | (data > hi), "#F44336", "#2196F3")
    ax.bar(new_designs["design_id"], data, color=color, alpha=0.7, width=0.8)
    ax.axhline(lo, color="green",  linestyle="--", linewidth=1.2, label=f"Min ({lo})")
    ax.axhline(hi, color="orange", linestyle="--", linewidth=1.2, label=f"Max ({hi})")
    ax.set_title(label, fontsize=10)
    ax.set_xlabel("Design ID", fontsize=8)
    ax.legend(fontsize=7)
    ax.grid(True, alpha=0.3)

plt.suptitle("Design Parameters vs Standard Bounds  (🔴 = Out-of-Range)", fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig("parameter_distribution.png", bbox_inches="tight")
plt.show()
print("Figure saved to parameter_distribution.png")


## 6. Automated Validation Report Generation

The report generator consolidates all rule violations and AI anomaly results  
into a single, printable validation report.


In [ ]:
# Attach AI results to engine so they appear in the report
engine._ai_results = ai_results

reporter = ValidationReportGenerator(engine, project_name="Mechanical Housing Assembly v2.1")
report_text = reporter.generate(print_report=True)

# Save report to file
with open("validation_report.txt", "w") as f:
    f.write(report_text)
print("\n📄 Report saved to validation_report.txt")


## 7. Compliance Score Dashboard

In [ ]:
summary = engine.summary()
total_checks = len(new_designs) * 3   # dims + material + assembly per design
total_issues = summary["total_issues"]
ai_anomalies = summary["ai_anomalies"]

# Compliance metrics
rule_compliance  = max(0, (total_checks - total_issues) / total_checks * 100)
ai_compliance    = (len(ai_results) - ai_anomalies) / len(ai_results) * 100
overall_score    = (rule_compliance * 0.6 + ai_compliance * 0.4)

sev_data = summary["by_severity"]

fig = plt.figure(figsize=(16, 9))
gs  = fig.add_gridspec(2, 3, hspace=0.4, wspace=0.35)

# ── Gauge: overall compliance score ──────────────────────────────────────
ax0 = fig.add_subplot(gs[0, 0])
wedge_colors = ["#4CAF50" if overall_score >= 80
                else "#FF9800" if overall_score >= 60
                else "#F44336", "#ECEFF1"]
wedge_sizes  = [overall_score, 100 - overall_score]
ax0.pie(wedge_sizes, colors=wedge_colors, startangle=90,
        wedgeprops=dict(width=0.4, edgecolor="white"))
ax0.text(0, 0, f"{overall_score:.1f}%", ha="center", va="center",
         fontsize=22, fontweight="bold",
         color="#4CAF50" if overall_score >= 80 else
               "#FF9800" if overall_score >= 60 else "#F44336")
ax0.set_title("Overall Compliance Score", pad=15, fontsize=11)

# ── Bar: rule vs AI compliance ────────────────────────────────────────────
ax1 = fig.add_subplot(gs[0, 1])
categories = ["Rule-Based\nCompliance", "AI-Based\nCompliance"]
values     = [rule_compliance, ai_compliance]
bar_colors = ["#2196F3", "#9C27B0"]
b = ax1.bar(categories, values, color=bar_colors, width=0.45, edgecolor="white")
ax1.bar_label(b, fmt="%.1f%%", padding=3)
ax1.set_ylim(0, 110)
ax1.set_ylabel("Compliance (%)")
ax1.set_title("Compliance by Check Type", fontsize=11)
ax1.axhline(80, color="green", linestyle="--", linewidth=1, alpha=0.6, label="Target 80%")
ax1.legend(fontsize=9)

# ── Stacked bar: severity breakdown ─────────────────────────────────────
ax2 = fig.add_subplot(gs[0, 2])
sev_labels = ["CRITICAL", "HIGH", "MEDIUM", "LOW"]
sev_vals   = [sev_data.get(s, 0) for s in sev_labels]
s_colors   = ["#D32F2F", "#F57C00", "#1976D2", "#0097A7"]
bars_s     = ax2.bar(["Issues"], [sum(sev_vals)],
                     color="white", edgecolor="gray", linewidth=0.5)
bottom = 0
for s, v, c in zip(sev_labels, sev_vals, s_colors):
    ax2.bar(["Issues"], [v], bottom=bottom, color=c, label=f"{s} ({v})")
    bottom += v
ax2.set_ylabel("Count")
ax2.set_title("Rule Issues by Severity", fontsize=11)
ax2.legend(loc="upper right", fontsize=9)

# ── Timeline: issues across components (top 20) ───────────────────────────
ax3 = fig.add_subplot(gs[1, :2])
if not issues_df.empty:
    comp_counts = (issues_df.groupby("component")["severity"]
                   .count()
                   .sort_values(ascending=False)
                   .head(20))
    sev_by_comp = (issues_df[issues_df["component"].isin(comp_counts.index)]
                   .groupby(["component","severity"])
                   .size()
                   .unstack(fill_value=0)
                   .reindex(comp_counts.index))
    sev_by_comp.plot(kind="bar", ax=ax3,
                     color={"CRITICAL":"#D32F2F","HIGH":"#F57C00",
                            "MEDIUM":"#1976D2","LOW":"#0097A7","INFO":"#388E3C"},
                     stacked=True, width=0.8, edgecolor="white")
    ax3.set_xlabel("")
    ax3.set_ylabel("Issue Count")
    ax3.set_title("Top 20 Components by Issue Count", fontsize=11)
    ax3.tick_params(axis="x", rotation=45, labelsize=8)
    ax3.legend(title="Severity", fontsize=8)
else:
    ax3.text(0.5, 0.5, "No rule issues detected ✅",
             ha="center", va="center", fontsize=14)

# ── Anomaly score timeline ─────────────────────────────────────────────
ax4 = fig.add_subplot(gs[1, 2])
c_map = ai_results["anomaly_label"].map({"NORMAL":"#2196F3","ANOMALY":"#F44336"})
ax4.barh(ai_results["design_id"], ai_results["anomaly_score"], color=c_map, height=0.7)
ax4.axvline(0, color="gray", linestyle="--", linewidth=1)
ax4.set_xlabel("Anomaly Score")
ax4.set_ylabel("Design ID")
ax4.set_title("AI Anomaly Scores", fontsize=11)
normal_p  = mpatches.Patch(color="#2196F3", label="Normal")
anomaly_p = mpatches.Patch(color="#F44336", label="Anomaly")
ax4.legend(handles=[normal_p, anomaly_p], fontsize=9)

fig.suptitle("AI-Integrated CAD Design Validation Dashboard", fontsize=15, y=1.01, fontweight="bold")
plt.savefig("compliance_dashboard.png", bbox_inches="tight")
plt.show()
print("Dashboard saved to compliance_dashboard.png")


## 8. Real-Time Single Component Validation (CAD Integration Simulation)

This section demonstrates how the system would work **inline within a CAD tool**:  
a designer submits a component specification and receives instant feedback.


In [ ]:
def realtime_cad_check(component_name, dimensions, material, assembly):
    """
    Simulates the real-time CAD validation hook that would be called
    whenever a designer modifies a part in the CAD environment.
    """
    rt_engine = CADValidationEngine()
    issues = rt_engine.validate_component(component_name, dimensions, material, assembly)

    print(f"\n{'='*60}")
    print(f"  REAL-TIME VALIDATION: {component_name}")
    print(f"{'='*60}")
    if not issues:
        print("  ✅ No issues detected — design complies with all standards.")
    else:
        for issue in issues:
            icon = {"CRITICAL":"🔴","HIGH":"🟠","MEDIUM":"🔵","LOW":"🟢"}.get(issue.severity,"ℹ️")
            print(f"  {icon} [{issue.severity}] {issue.parameter}: "
                  f"{issue.measured_value} → {issue.description}")
            print(f"      💡 {issue.suggestion}")
    print(f"{'='*60}")
    return issues

# ── Test 1: Compliant design ──────────────────────────────────────────────
realtime_cad_check(
    "BracketA",
    dimensions={"length_mm":100, "width_mm":50, "wall_thickness_mm":5,
                "hole_diameter_mm":10, "fillet_radius_mm":2,
                "draft_angle_deg":2, "surface_roughness_um":1.6},
    material={"name":"Steel","yield_strength_mpa":350,"density_g_cm3":7.8},
    assembly={"interference_mm":0.02,"clearance_mm":0.15,"component_count":120},
)

# ── Test 2: Non-compliant design ─────────────────────────────────────────
realtime_cad_check(
    "CoverPlateX",
    dimensions={"length_mm":5200, "width_mm":50, "wall_thickness_mm":0.3,
                "hole_diameter_mm":0.2, "fillet_radius_mm":2,
                "draft_angle_deg":2, "surface_roughness_um":28},
    material={"name":"Plastic","yield_strength_mpa":60,"density_g_cm3":1.2},
    assembly={"interference_mm":0.08,"clearance_mm":0.04,"component_count":600},
)


## 9. System Architecture Summary

```
┌─────────────────────────────────────────────────────────────────────┐
│               AI-Integrated CAD Design Validation System             │
├───────────────────┬─────────────────────────────────────────────────┤
│  CAD Environment  │  Design parameters fed in real-time or in batch  │
│  (Integration)    │  via validate_component() / validate_batch()     │
├───────────────────┴─────────────────────────────────────────────────┤
│                     CADValidationEngine                              │
│  ┌────────────────────────┐   ┌────────────────────────────────────┐ │
│  │ DesignStandardsChecker │   │      AIAnomalyDetector             │ │
│  │  • Dimension bounds    │   │  • Isolation Forest model          │ │
│  │  • Material rules      │   │  • Trained on compliant history    │ │
│  │  • Assembly constraints│   │  • Flags statistical outliers      │ │
│  └────────────────────────┘   └────────────────────────────────────┘ │
├──────────────────────────────────────────────────────────────────────┤
│                  ValidationReportGenerator                            │
│  • Severity-ranked issue list  • Actionable suggestions              │
│  • Compliance score dashboard  • Exportable report                   │
└──────────────────────────────────────────────────────────────────────┘
```

### Key Benefits
| Feature | Benefit |
|---|---|
| Rule-based validation | Instant, deterministic compliance checks |
| AI anomaly detection | Catches subtle deviations not covered by rules |
| Severity classification | Prioritises critical issues for early action |
| Actionable suggestions | Reduces rework by guiding corrective steps |
| Report generation | Audit trail and compliance documentation |
